# Doubly Robust Estimation of Causal Effects Under Unconfoundness With EconML

## Set Project Root

In [ ]:
from pathlib import Path

def find_project_root(start: Path | None = None, marker: str = "pyproject.toml") -> Path:
    """
    Find the project root by walking upward from `start` until `marker` is found.

    Used only to resolve repo-relative paths such as evals/, configs/, or data/.
    """
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate

    raise RuntimeError(
        f"Could not locate project root: no {marker!r} found in {start} or its parents. "
        "Run this notebook from inside the repository, or update PROJECT_ROOT manually."
    )

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = find_project_root()

## Configuration

In [ ]:
SEED = 1024
COVARIATES = [f"x_{i + 1}" for i in range(58)]

ACIC_TREATMENT = 1
ACIC_RESPONSE = 13

# Adds squares and bins continuous covariates to help achieve balance:
AUGMENT_CONTINUOUS_COVARIATES = False

# Trims extreme propensity scores for purposes of calculating the ATO.
ATO_THRESHOLD = 0.1

# Decide which estimand to return.
ESTIMATE_ATE = True
ESTIMATE_ATT = True
ESTIMATE_ATO = True

# Max threads per XGBoost model (set by scheduler based on parallelism).
N_JOBS = -1


In [ ]:
assert ATO_THRESHOLD > 0.0, "ATO threshold must be positive."
assert ATO_THRESHOLD < 0.5, "ATO threshold must be < 0.5."

In [ ]:
assert any([ESTIMATE_ATE, ESTIMATE_ATT, ESTIMATE_ATO]), "Must estimate at least one estimand."

## Dependencies

In [ ]:
import copy

import numpy as np
import pandas as pd

In [ ]:
from oci_agent.backends.estimators import (
    XGBEarlyStoppingRegressor,
    XGBEarlyStoppingClassifier,
)
from oci_agent.backends.utils import (
    make_numeric_ml_dataframe,
    summarize,
    standardized_mean_differences,
    weighted_standardized_mean_differences,
    is_probably_continuous,
    trim_by_propensity,
    att_ipw_weights,
)
from oci_agent.backends.econml_helpers import (
    make_nuisance_design_matrix,
    extract_feature_importances_from_econml_models,
    extract_avg_propensity_scores,
    extract_avg_outcome_predictions,
    aipw_pseudo_outcome,
    att_dr_pseudo_outcome,
    aipw_summary,
)

In [ ]:
from econml.dr import DRLearner
from sklearn.linear_model import LinearRegression
from xgboost import XGBClassifier, XGBRegressor

## Declare Estimators

In [ ]:
import yaml as _yaml

_xgb_cfg = _yaml.safe_load((PROJECT_ROOT / "configs" / "xgboost.yaml").read_text())
_shared = _xgb_cfg["shared"]
_es = _xgb_cfg["early_stopping"]

xgb_y_base = XGBRegressor(
    **_shared,
    objective=_xgb_cfg["outcome"]["objective"],
    n_jobs=N_JOBS,
)

xgb_t_base = XGBClassifier(
    **_shared,
    objective=_xgb_cfg["propensity"]["objective"],
    n_jobs=N_JOBS,
)

model_regression = XGBEarlyStoppingRegressor(
    base_estimator=xgb_y_base,
    validation_fraction=_es["validation_fraction"],
    early_stopping_rounds=_es["early_stopping_rounds"],
    eval_metric=_es["outcome_eval_metric"],
    random_state=_es["random_state"],
)

model_propensity = XGBEarlyStoppingClassifier(
    base_estimator=xgb_t_base,
    validation_fraction=_es["validation_fraction"],
    early_stopping_rounds=_es["early_stopping_rounds"],
    eval_metric=_es["propensity_eval_metric"],
    random_state=_es["random_state"],
    stratify=_es["propensity_stratify"],
)


## Load Data

In [ ]:
def get_dataset(treatment_ix, response_ix, root=None):
    root = Path(root) if root is not None else PROJECT_ROOT / "evals" / "acic2016"
    covariates = pd.read_csv(root / "x.csv")
    treatment_and_response = pd.read_csv(root / str(treatment_ix) / f"zymu_{response_ix}.csv")
    response = pd.Series(
        np.where(
            treatment_and_response["z"] == 1,
            treatment_and_response["y1"],
            treatment_and_response["y0"],
        ),
        name="y"
    )
    return pd.concat([covariates, treatment_and_response["z"], response], axis=1)


In [ ]:
dataset = get_dataset(ACIC_TREATMENT, ACIC_RESPONSE)

In [ ]:
dataset.head()

In [ ]:
dataset.shape

## Prepare Analysis Dataset

In [ ]:
Y = dataset["y"].to_numpy()
T = dataset["z"].to_numpy().astype(int)
X = make_numeric_ml_dataframe(dataset[COVARIATES])

In [ ]:
if AUGMENT_CONTINUOUS_COVARIATES:
    for col in X.columns:
        if is_probably_continuous(X[col]):
            X[f"{col}__sq"] = X[col] ** 2

            X[f"{col}__qbin"] = pd.qcut(
                X[col],
                q=5,
                labels=False,
                duplicates="drop"
            )

## Estimation

In [ ]:
est = DRLearner(
    model_propensity=model_propensity,
    model_regression=model_regression,
    model_final=LinearRegression(),
    cv=5,
    random_state=SEED,
)

est.fit(Y, T, X=X, W=None)

## Diagnostics

### Diagnostic 1. Check Covariate Balance Before and After Weighting

In [ ]:
# Get propensity scores (out-of-fold).
ps = extract_avg_propensity_scores(est, T, X=X, W=None, treatment_value=1)
ipw = np.where(
    T == 1,
    1 / ps,
    1 / (1 - ps)
)

In [ ]:
def get_important_features(est, X, W, threshold=10):
    Z = make_nuisance_design_matrix(X=X, W=W)
    prop_importance = extract_feature_importances_from_econml_models(
        est.models_propensity,
        feature_names=Z.columns
    )
    prop_features = prop_importance["feature"].head(threshold)

    outcome_importance = extract_feature_importances_from_econml_models(
        est.models_regression,
        feature_names=[*Z.columns, "T"]
    )
    outcome_features = outcome_importance["feature"].head(threshold)

    intersection = list(set(prop_features).intersection(outcome_features))
    if intersection:
        return intersection
    # Fall back to the union when the top-K sets are disjoint — keeps balance
    # measurable even when propensity and outcome models prioritize different
    # features. Restrict to Z columns since "T" appears only among outcome
    # features and is not part of the design matrix used for balance.
    return [f for f in set(prop_features).union(outcome_features) if f in Z.columns]


In [ ]:
def summarize_covariate_balance(est, weights, T, X=None, W=None):
    important_features = get_important_features(est, X, W)
    Z = make_nuisance_design_matrix(X=X, W=W)
    balance_unweighted = standardized_mean_differences(
        Z,
        treatment=T,
        covariates=important_features,
    )
    balance_weighted = weighted_standardized_mean_differences(
        Z,
        treatment=T,
        weights=weights,
        covariates=important_features
    )
    return (
        balance_unweighted[["feature", "smd", "abs_smd"]]
        .merge(
            balance_weighted[["feature", "weighted_smd", "abs_weighted_smd"]],
            on="feature",
            how="left"
        )
        .sort_values("abs_weighted_smd", ascending=False)
    )

### Covariate Balance for ATE

In [ ]:
ate_covariate_balance = summarize_covariate_balance(est, ipw, T, X=X)
ate_covariate_balance

### Diagnostic 2. Check Propensity Score Overlap

In [ ]:
overlap = pd.DataFrame({
    "T": T,
    "propensity_score": ps
})

overlap.groupby("T")["propensity_score"].describe()

### Diagnostic 3. Treatment Effect on Placebo

In [ ]:
def get_pseudo_outcome(est, T, X, Y, return_att=False):
    """AIPW doubly-robust pseudo-outcome (out-of-fold nuisances)."""
    ps = extract_avg_propensity_scores(est, T, X=X, W=None, treatment_value=1)
    m0_hat = extract_avg_outcome_predictions(est, T, X=X, treatment_value=0)
    m1_hat = extract_avg_outcome_predictions(est, T, X=X, treatment_value=1)
    if return_att:
        return att_dr_pseudo_outcome(Y, T, m0_hat, ps)
    return aipw_pseudo_outcome(Y, T, m0_hat, m1_hat, ps)

In [ ]:
def run_placebo_test(est, T, X, return_att=False):
    """Estimate the 'treatment effect' on the most important pre-treatment
    covariate for explaining the outcome. The placebo's true effect is zero
    by construction; a significant estimate signals residual confounding."""

    # Pick the most important outcome feature (excluding the treatment T itself)
    # as the placebo outcome.
    Z = make_nuisance_design_matrix(X=X, W=None)
    outcome_importance = extract_feature_importances_from_econml_models(
        est.models_regression,
        feature_names=[*Z.columns, "T"]
    )
    placebo = (
        outcome_importance[outcome_importance.feature != "T"]
        .feature.head(1).iloc[0].replace("X__", "")
    )

    Y_placebo = X[placebo]
    X_placebo = X[[c for c in X.columns if c != placebo]]

    placebo_est = copy.deepcopy(est)
    placebo_est.fit(Y_placebo, T, X=X_placebo, W=None)
    # Use Y_placebo (the refitted outcome) — not the main Y — when assembling
    # the DR pseudo-outcome for the placebo.
    return get_pseudo_outcome(placebo_est, T, X_placebo, Y_placebo, return_att)


In [ ]:
if ESTIMATE_ATE:
    placebo_ate_hat, placebo_ate_stderr, placebo_ate_ci = summarize(run_placebo_test(est, T, X))
    print(f'Placebo: {placebo_ate_hat:.4f} (SE={placebo_ate_stderr:.4f}, CI=[{placebo_ate_ci[0]:.4f}, {placebo_ate_ci[1]:.4f}])')

## Results

### Average Treatment Effect (ATE)

In [ ]:
Y_DR = get_pseudo_outcome(est, T, X, Y)

if ESTIMATE_ATE:
    ate_hat, ate_stderr, ate_ci = aipw_summary(Y_DR)
    print(
        f"ATE: {ate_hat:.3f}, "
        f"Std. Err: {ate_stderr:.3f}, "
        f"95% CI: ({ate_ci[0]:.3f}, {ate_ci[1]:.3f})"
    )

### Average Treatment Effect on the Treated (ATT)

In [ ]:
if ESTIMATE_ATT:
    # AIPW ATT estimator. The efficient influence function for ATT is
    #   psi_i = (1/pi) [T_i (Y_i - m0 - tau) - (1-T_i) (e/(1-e)) (Y_i - m0)].
    # centering=T/pi subtracts the T*tau/pi term that accounts for estimating
    # pi=E[T]; without it the SE is wider by O(tau^2 (1-pi) / (pi N)).
    Y_DR_ATT = get_pseudo_outcome(est, T, X, Y, return_att=True)
    _pi = float(np.mean(T))
    att_hat, att_stderr, att_ci = aipw_summary(Y_DR_ATT, centering=T / _pi)
    print(
        f"ATT: {att_hat:.3f}, "
        f"Std. Err: {att_stderr:.3f}, "
        f"95% CI: ({att_ci[0]:.3f}, {att_ci[1]:.3f})"
    )

    print("\nCovariate Balance for ATT:")
    att_ipw = att_ipw_weights(ps, T)
    att_covariate_balance = summarize_covariate_balance(est, att_ipw, T, X=X)
    with pd.option_context('display.float_format', '{:.3f}'.format):
        print(att_covariate_balance)
    placebo_att_hat, placebo_att_stderr, placebo_att_ci = aipw_summary(
        run_placebo_test(est, T, X, return_att=True),
        centering=T / _pi,
    )
    print(
        f"Placebo ATT: {placebo_att_hat:.3f}, "
        f"Placebo Std. Err: {placebo_att_stderr:.3f}, "
        f"Placebo 95% CI: ({placebo_att_ci[0]:.3f}, {placebo_att_ci[1]:.3f})"
    )

### Average Treatment Effect on the Overlap Population (ATO)

In [ ]:
if ESTIMATE_ATO:
    keep = trim_by_propensity(ps, T, lower=ATO_THRESHOLD, upper=1 - ATO_THRESHOLD)
    # AIPW with sample restricted to the trimmed-overlap subset.
    ato_hat, ato_stderr, ato_ci = aipw_summary(Y_DR[keep])
    print(
        f"ATO: {ato_hat:.3f}, "
        f"Std. Err: {ato_stderr:.3f}, "
        f"95% CI: ({ato_ci[0]:.3f}, {ato_ci[1]:.3f})"
    )

    print("\nCovariate Balance for ATO:")
    ato_covariate_balance = summarize_covariate_balance(est, ipw[keep], T[keep], X=X[keep])
    with pd.option_context('display.float_format', '{:.3f}'.format):
        print(ato_covariate_balance)
    placebo_ato_hat, placebo_ato_stderr, placebo_ato_ci = summarize(
        run_placebo_test(est, T, X)[keep]
    )
    print(
        f"Placebo ATO: {placebo_ato_hat:.3f}, "
        f"Placebo Std. Err: {placebo_ato_stderr:.3f}, "
        f"Placebo 95% CI: ({placebo_ato_ci[0]:.3f}, {placebo_ato_ci[1]:.3f})"
    )

In [ ]:
# === INJECTED BY OCI AGENT === #
import numpy as _np
import pandas as pd
from scipy.stats import norm as _norm


def _balance_dict(df):
    """Compact JSON-safe summary of a per-covariate SMD DataFrame."""
    abs_smd = df["abs_weighted_smd"]
    return {
        "max_abs_weighted_smd": float(abs_smd.max()),
        "fraction_above_0_2": float((abs_smd > 0.2).mean()),
        "per_covariate": {
            str(k): float(v)
            for k, v in df.set_index("feature")["weighted_smd"].items()
        },
    }


def _overlap_dict(ps_array, mask=None):
    p = pd.Series(ps_array)
    if mask is not None:
        p = p[mask]
    return {
        "fraction_outside_0_1_0_9": float(((p < 0.1) | (p > 0.9)).mean()),
        "propensity_min": float(p.min()),
        "propensity_max": float(p.max()),
        "propensity_mean": float(p.mean()),
        "n": int(len(p)),
    }


def _placebo_dict(hat, stderr, ci):
    z = hat / stderr if stderr else float("nan")
    p_value = float(2 * (1 - _norm.cdf(abs(z)))) if stderr else float("nan")
    return {
        "effect": float(hat),
        "stderr": float(stderr),
        "ci_lower": float(ci[0]),
        "ci_upper": float(ci[1]),
        "p_value": p_value,
    }


ate_result = None
if ESTIMATE_ATE:
    ate_result = {
        "estimate": float(ate_hat),
        "stderr": float(ate_stderr),
        "ci_lower": float(ate_ci[0]),
        "ci_upper": float(ate_ci[1]),
        "diagnostics": {
            "covariate_balance": _balance_dict(ate_covariate_balance),
            "overlap": _overlap_dict(ps),
            "placebo": _placebo_dict(placebo_ate_hat, placebo_ate_stderr, placebo_ate_ci),
        },
    }

att_result = None
if ESTIMATE_ATT:
    att_result = {
        "estimate": float(att_hat),
        "stderr": float(att_stderr),
        "ci_lower": float(att_ci[0]),
        "ci_upper": float(att_ci[1]),
        "diagnostics": {
            "covariate_balance": _balance_dict(att_covariate_balance),
            "overlap": _overlap_dict(ps, mask=(T == 1)),
            "placebo": _placebo_dict(placebo_att_hat, placebo_att_stderr, placebo_att_ci),
        },
    }

# `keep_indices` for the ATO trimmed sample is exposed as a top-level
# variable rather than embedded in ato_result. The runner writes it to a
# sibling `keep_indices.json` so it does not bloat results.json and crowd out
# downstream diagnostics when the critic reads a truncated view.
ato_keep_indices = None
ato_result = None
if ESTIMATE_ATO:
    ato_keep_indices = [int(i) for i in _np.where(_np.asarray(keep))[0]]
    ato_result = {
        "estimate": float(ato_hat),
        "stderr": float(ato_stderr),
        "ci_lower": float(ato_ci[0]),
        "ci_upper": float(ato_ci[1]),
        "diagnostics": {
            "covariate_balance": _balance_dict(ato_covariate_balance),
            "overlap": _overlap_dict(ps, mask=keep),
            "placebo": _placebo_dict(placebo_ato_hat, placebo_ato_stderr, placebo_ato_ci),
        },
    }
